In [1]:
import pandas as pd
import sys
import os

from nltk.corpus import stopwords
from sklearn.ensemble import AdaBoostClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier

if "Desktop" in os.getcwd():
    base_path = "/Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification"
else:
    base_path = "/home/sakdag/weakly-supervised-fake-news-classification"

print(base_path)
sys.path.append(base_path)
    
from src.utils.path_utils import get_project_root
from src.preprocessing.consts import Datasets
from src.preprocessing.read_raw_data import read_raw_data
from src.preprocessing.read_raw_data import read_csv
from src.weak_labeling.consts.globals import NLP

/Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification


In [2]:
injected_df: pd.DataFrame = read_csv(Datasets.NELA_COVID_2021.injected_path)

Reading dataset from path: /Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-covid-2021-sampled.csv
Read dataset, dataframe size: 50000


In [5]:
from src.weak_labeling.model_based.adaboost.adaboost import (
    train_adaboost_classifier,
    get_adaboost_labels,
)

In [6]:
adaboost_classifier, tfidf_vectorizer = train_adaboost_classifier(
    train_with=Datasets.NELA_COVID_2021
)

Reading dataset from path: /Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification/data/raw/nela/nela-covid-2021-sampled.csv
Read dataset, dataframe size: 209138
accuracy: 0.783


In [3]:
stop = stopwords.words("english")

injected_df["title_without_stopwords"] = injected_df["title"].apply(
    lambda x: " ".join([word for word in x.split() if word not in (stop)])
)
injected_df["content_without_stopwords"] = injected_df["content"].apply(
    lambda x: " ".join([word for word in x.split()[:200] if word not in (stop)])
)

y = injected_df.label
injected_df["x_train"] = injected_df["title_without_stopwords"] + injected_df["content_without_stopwords"]
x = injected_df["x_train"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20)

tfidf_vect = TfidfVectorizer(stop_words="english")
tfidf_train = tfidf_vect.fit_transform(x_train)
tfidf_test = tfidf_vect.transform(x_test)

abc = AdaBoostClassifier(base_estimator=DecisionTreeClassifier())

# parameters = {
#     "base_estimator__max_depth": [i for i in range(2, 11, 2)],
#     "base_estimator__min_samples_leaf": [5, 10],
#     "n_estimators": [10, 50, 250, 1000],
#     "learning_rate": [0.01, 0.1],
# }
parameters = {
    "base_estimator__max_depth": [2],
    "base_estimator__min_samples_leaf": [5],
    "n_estimators": [10, 50],
    "learning_rate": [0.1],
}

grid_search = GridSearchCV(abc, parameters, verbose=3, scoring="f1", n_jobs=-1)
results = grid_search.fit(tfidf_train, y_train)

print(results)
print(grid_search.best_estimator_)

Fitting 5 folds for each of 2 candidates, totalling 10 fits
GridSearchCV(estimator=AdaBoostClassifier(base_estimator=DecisionTreeClassifier()),
             n_jobs=-1,
             param_grid={'base_estimator__max_depth': [2],
                         'base_estimator__min_samples_leaf': [5],
                         'learning_rate': [0.1], 'n_estimators': [10, 50]},
             scoring='f1', verbose=3)
AdaBoostClassifier(base_estimator=DecisionTreeClassifier(max_depth=2,
                                                         min_samples_leaf=5),
                   learning_rate=0.1)


In [8]:
print(grid_search.best_params_)
print(grid_search.best_estimator_.base_estimator_.max_depth)

{'base_estimator__max_depth': 2, 'base_estimator__min_samples_leaf': 5, 'learning_rate': 0.1, 'n_estimators': 50}
2


In [ ]:
Adab = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=10), n_estimators=5, random_state=1
)